<a href="https://colab.research.google.com/github/leejunho12316/HonGongMachine/blob/main/3_2_%EB%A9%80%ED%8B%B0%ED%84%B4_%EC%BF%BC%EB%A6%AC_%EC%83%9D%EC%84%B1%EA%B8%B0_(26_02_22_%EA%B0%95%EC%9D%98%EC%9A%A9).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U langchain langchain-core langchain-community langchain-openai langchain-text-splitters pypdf faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.7/502.7 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.4/331.4 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.5 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.15
    Uninstalling langchain-core-1.2.15:
      Successfully uninstalled langchain-core-1.2.15
ERROR: pip's dependenc

In [ ]:
# OpenAI API 키 설정
import os
os.environ["OPENAI_API_KEY"] = None

#RAG 1

##데이터 불러오기/VectorDB 저장

In [ ]:
import urllib.request
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

urllib.request.urlretrieve("https://github.com/chatgpt-kr/openai-api-tutorial/raw/main/ch06/2023_%EB%B6%81%ED%95%9C%EC%9D%B8%EA%B6%8C%EB%B3%B4%EA%B3%A0%EC%84%9C.pdf", filename="2023_북한인권보고서.pdf")
loader = PyPDFLoader('2023_북한인권보고서.pdf')

doc_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=100)
docs = loader.load_and_split(doc_splitter)

embedding = OpenAIEmbeddings(model="text-embedding-3-large")
faiss_store = FAISS.from_documents(docs, embedding)

persist_directory = "/content/DB"
faiss_store.save_local(persist_directory)
vectordb = FAISS.load_local(persist_directory, embeddings=embedding, allow_dangerous_deserialization=True)


##이전 대화를 고려한 쿼리 재생성

모호한 후속 질문을 이전 대화 맥락을 활용해 구체적이고 명확한 검색 쿼리로 변환하는 class

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

class QueryRewriter:
  def __init__(self, model_name: str = "gpt-4.1", temperature : int = 0):
    self.llm = ChatOpenAI(model = model_name, temperature = temperature, max_tokens = 1000)
    self.prompt_template = ChatPromptTemplate.from_template("""
    당신은 사용자의 이전 대화와 현재 질문을 기반으로 검색 쿼리를 재생성하는 전문가입니다.

    # 지침
    1. 이전 대화 맥락과 현재 질문을 고려하여 더 정확한 검색 쿼리를 생성하세요.
    2. 현재 질문이 간략하거나 이전 대화의 맥락을 참조하는 경우, 이전 대화를 고려하여 완전한 쿼리를 구성하세요.
    3. 이전 대화가 없거나 관련이 없는 경우 현재 질문만 사용하세요.
    4. 반드시 명확하고 검색에 적합한 쿼리를 생성하세요.
    5. 출력은 재생성된 쿼리 문자열만 포함해야 합니다. 추가 설명이나 주석은 포함하지 마세요.

    # 입력
    이전 대화 질문: {prev_query}
    이전 대화 답변: {prev_response}
    현재 질문: {current_query}

    # 출력
    재생성된 쿼리:
""")

  def rewrite_query(self, prev_query : str, prev_response : str, current_query : str):
    if not prev_query.strip():
      return current_query

    prompt = self.prompt_template.format_messages(
        prev_query = prev_query,
        prev_response = prev_response,
        current_query = current_query
    )

    response = self.llm.invoke(prompt)

    return response.content.strip()


In [ ]:
query_writer= QueryRewriter()

answer = query_writer.rewrite_query(
    prev_query = "한국의 관광지 추천해줘",
    prev_response = "한국의 유명한 관광지로는 코타키나발루와 말레이시아의 코딱지가 있습니다",
    current_query = "거기 PC방은?"
)

print(answer)

코타키나발루와 말레이시아의 PC방 정보


##이전 대화를 고려한 쿼리 재생성 + 검색기
- 최근 대화 업데이트
- 업데이트 된 최근 대화와 QueryRewriter로 쿼리 재작성
- 재작성된 쿼리로 VectorDB 검색

In [ ]:
class ConversationalRetriever:
  def __init__(self, vectorstore, query_rewriter, prev_query:str = "", prev_response:str = ""):
    self.vectorstore = vectorstore
    self.query_rewriter = query_rewriter

    self.prev_query = prev_query
    self.prev_response = prev_response

  #최근 대화 업데이트
  def update_conversation(self, prev_query : str, prev_response : str):
    self.prev_query = prev_query
    self.prev_response = prev_response

  #QueryRewriter로 쿼리 재작성 후 VectorDB 검색
  def get_relevant_documents(self, current_query : str, num_docs: int = 4):
    rewritten_query = self.query_rewriter.rewrite_query(
        self.prev_query, self.prev_response, current_query
    )

    print('QueryRewriter')
    print(f'현재 query : {current_query}')
    print(f'재작성 query : {rewritten_query}')
    print('-' * 50)

    docs = self.vectorstore.similarity_search(rewritten_query, search_kwargs = {'k' : num_docs})

    return docs



##두 클래스를 사용한 RAG 시스템 완성

In [ ]:
class ConversationalRAG:
  def __init__(self, vectorstore, model_name : str = 'gpt-4.1', temperature : float = 0.2):
    self.query_rewriter = QueryRewriter(model_name = model_name, temperature = temperature)

    self.retriever = ConversationalRetriever(vectorstore, self.query_rewriter)

    self.llm = ChatOpenAI(model = model_name, temperature = temperature)

    self.prompt_template = ChatPromptTemplate.from_template("""
        당신은 문서 기반 질문 응답 시스템입니다.
        아래의 context를 바탕으로 질문에 대답하세요.
        만약 context에 관련 정보가 없으면 '해당 정보는 제공된 문서에 없습니다.' 라고 답하세요.

        질문: {question}

        context:
        {context}

        답변:
    """)

  def query(self, current_query : str, num_docs : int = 4):
    #1. 문서 검색
    docs = self.retriever.get_relevant_documents(current_query, num_docs)
    context = "\n\n".join([doc.page_content for doc in docs])
    print("Retriever 문서 검색")
    print(context)
    print('-' * 50)

    #2. 답변 생성
    prompt = self.prompt_template.format_messages(
        question = current_query,
        context = context
    )
    response = self.llm.invoke(prompt)
    print("Retriever LLM 답변 생성")
    print(response)
    print('-' * 50)


    #3. Retriever 대화 업데이트
    self.retriever.update_conversation(current_query, response.content)
    print("Retriever 대화 업데이트")
    print('-' * 50)

    #4. 결과 반환
    return {
        'question' : current_query,
        'response' : response,
        'source_documents' : docs
    }

실행

In [ ]:
rag = ConversationalRAG(vectordb, 'gpt-4.1', 0.2)

In [ ]:
answer = rag.query('19년 말 평양시 소재 기업소에서 달마다 배급받은 음식')
print(f'질문 : {answer['question']}')
print(f'답변 : {answer['response'].content}')

QueryRewriter
현재 query : 19년 말 평양시 소재 기업소에서 달마다 배급받은 음식
재작성 query : 19년 말 평양시 소재 기업소에서 달마다 배급받은 음식
--------------------------------------------------
Retriever 문서 검색
화 또는 쌀이나 기름 등 현물로 지급하였다고 한다. 2019년 평양
의 외화벌이 사업소에서는 보수 50달러를 월 2회로 나누어 현금으
로 지급하였다고 하는 사례가 있었고, 평양 외화벌이 식당에서는 매

파악되었다. 따라서 기관·기업소의 상황에 따라 식량배급량, 주기, 
곡식종류에 상당한 차이가 있는 것으로 나타났다. 외화벌이 기관 등
에는 식량배급이 원활하게 이뤄지고 있었다는 증언이 수집되었다. 
2019년 평양시에서 기업소 운전원으로 일하였던 노동자는 매월 쌀·
설탕·기름·야채·돼지고기 등을 배급받아 식량이 부족하지 않았다는 
증언과 2019년 중앙당 산하의 기업소에서 매월 쌀 6㎏ 정도, 기름 5
ℓ, 설탕 2㎏, 맛내기 2봉지, 돼지고기 2㎏, 닭고기 1마리 정도 받았

가배급을 선택하고, 잘사는 기업소들은 기업소 자체 배급을 선택합
니 다. 세대주가 직장에 다닐 경우 세대주만 직장에서 배급을 받고 
가족들은 국가배급소에서 배급을 받습니다. 평양시와 자강도는 대
체로 다 줬는데 다른 지역은 배급이 잘 안되고 배급제가 없어졌다는 
소리를 들었습니다. ”
국가배급의 주기, 양, 곡물의 종류 등에서 평양시와 지방의 차이
가 크게 나고 있었다. 식량배급이 비교적 원활하게 작동하는 지역은 
평양시로 보이는데, 2017년 어머니가 지역배급 대상자로 배급표가

한 달을 생활하기에 부족한 금액이었다고 하였다. 2018년 양강도의 
무역사업소에서는 1년치 노동 보수와 배급을 한 번에 지급하였다고 
하는데, 지급된 금액은 노동자 1명에게 1,800위안으로 약 300만원 
정도였다고 하였다. 2019년 양강도의 합영회사는 노동자에게 매달 
9~12만원의 보수를 지급하고, 1년에 한번

In [ ]:
answer = rag.query('식량이 충분했는지')
print(f'질문 : {answer['question']}')
print(f'답변 : {answer['response'].content}')

QueryRewriter
현재 query : 식량이 충분했는지
재작성 query : 2019년 평양시 기업소 노동자들이 배급받은 식량이 생활에 충분했는지 증언이나 평가
--------------------------------------------------
Retriever 문서 검색
파악되었다. 따라서 기관·기업소의 상황에 따라 식량배급량, 주기, 
곡식종류에 상당한 차이가 있는 것으로 나타났다. 외화벌이 기관 등
에는 식량배급이 원활하게 이뤄지고 있었다는 증언이 수집되었다. 
2019년 평양시에서 기업소 운전원으로 일하였던 노동자는 매월 쌀·
설탕·기름·야채·돼지고기 등을 배급받아 식량이 부족하지 않았다는 
증언과 2019년 중앙당 산하의 기업소에서 매월 쌀 6㎏ 정도, 기름 5
ℓ, 설탕 2㎏, 맛내기 2봉지, 돼지고기 2㎏, 닭고기 1마리 정도 받았

정도의 옥수수를 배급받았다는 진술과 2018년 평양시 기업소에서  
1년에 한번 배급하였는데, 규정보다 적은 양의 옥수수로 제공되어 가
족 4명이 2~3개월 정도 생활할 수 있는 양이었다고 하였다. 2017년 
평양시 소재 기업소는 노동자에게 한번 1개월분의 식량을 배급하였
으며, 1인당 2㎏ 정도만 지급되었다는 진술이 있었다. 2018년 양강
도 철도노동자에게 직장노동자 700g으로 규정대로 배급표가 나왔
지만, 실제 식량배급은 1년 동안 감자 150㎏을 한번 받았다는 사례

1. 식량권
253
IV. 경제적·사회적·문화적 권리 I. 발간개요V. 취약계층VI. 특별사안 II. 요약III. 시민적·정치적 권리
근무하는 동안 배급을 한 번도 받아본 적이 없었다고 진술하였고, 
2018년 함경북도의 광산기업소가 운영이 되지 않아 노동자에 대한 
배급도 전혀 지급되지 않았다는 증언도 수집되었다. 근무하던 곳에
서 2017년 즈음부터 배급되지 않았고, 2019년 군당 소속 기관에서 
일하였지만 배급은 지급되지 않았다는 진술도 있었다. 
기관·기업소에서는 노동자에게 배급할 식량을 직접 생산하도록

노동자는 노동의

In [ ]:
import json
from typing import List, Dict, Any, Tuple, Optional

from pydantic import BaseModel, Field

from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.documents import Document
from langchain_core.output_parsers import JsonOutputParser

#RAG 2

In [4]:
# OpenAI API 키 설정
import os
os.environ["OPENAI_API_KEY"] = None

데이터 다운로드, VectorDB 적재

In [5]:
import urllib.request
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

urllib.request.urlretrieve("https://github.com/chatgpt-kr/openai-api-tutorial/raw/main/ch06/2023_%EB%B6%81%ED%95%9C%EC%9D%B8%EA%B6%8C%EB%B3%B4%EA%B3%A0%EC%84%9C.pdf", filename="2023_북한인권보고서.pdf")
loader = PyPDFLoader('2023_북한인권보고서.pdf')

doc_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=100)
docs = loader.load_and_split(doc_splitter)

embedding = OpenAIEmbeddings(model="text-embedding-3-large")

faiss_store_mulit_query = FAISS.from_documents(docs, embedding)
persist_directory = "/content/multi_quiery_DB"
faiss_store_mulit_query.save_local(persist_directory)

vectordb = FAISS.load_local(persist_directory, embeddings=embedding, allow_dangerous_deserialization=True)

pydantic BaseModel, json_parser

In [6]:
from pydantic import BaseModel, Field
from typing import List
import json
from langchain_core.output_parsers import JsonOutputParser

class MultiQueryGenerator(BaseModel):
  queries : List[str] = Field(description = '사용자 질문에서 추출한 검색 쿼리 목록')

def json_parser(response):
  response_json = json.loads(response.content)

  try:
    if 'queries' in response_json and isinstance(response_json['queries'], list):
      return response_json['queries']
    else:
      print(f'응답에 queries 리스트가 없습니다 : {response_json}')
      return None
  except json.JSONDecodeError as e:
      print(f'JSON 파싱 오류 : {str(e)}')
      return None

##멀티턴 멀티쿼리 생성기

In [7]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

class AdvancedQueryRewriter:
  def __init__(self, model_name : str = 'gpt-4.1', temperature : float = 0):
    self.llm = ChatOpenAI(model = model_name, temperature = temperature, max_tokens = 1000)
    self.parser = JsonOutputParser(pydantic_object = MultiQueryGenerator)

    self.prompt_template = ChatPromptTemplate.from_template("""
    당신은 사용자의 이전 대화와 현재 질문을 기반으로 검색 쿼리를 재생성하는 전문가입니다.

    # 지침
    1. 이전 대화 맥락과 현재 질문을 고려하여 더 정확한 검색 쿼리를 생성하세요.
    2. 현재 질문이 간략하거나 이전 대화의 맥락을 참조하는 경우, 이전 대화를 고려하여 완전한 쿼리를 구성하세요.
    3. 이전 대화가 없거나 관련이 없는 경우 현재 질문만 사용하세요.
    4. 현재 질문에 여러 개의 질의가 포함되어 있다면, 각각을 별도의 쿼리로 분리하세요.
    5. 각 쿼리는 독립적으로 벡터 DB에서 검색될 수 있도록 완전하고 명확해야 합니다.
    6. 마크다운 형식으로 작성하지 마십시오.

    # 입력
    이전 대화 질문: {prev_query}
    이전 대화 답변: {prev_response}
    현재 질문: {current_query}

    {format_instructions}
    """)

  def rewrite_query(self, prev_query : str, prev_response : str, current_query : str):
    if not prev_query.strip():
      return [current_query]

    try:
      #1. LLM 응답 받기 (JSON 포맷 지정)
      format_instructions = self.parser.get_format_instructions()

      prompt = self.prompt_template.format_messages(
          prev_query = prev_query,
          prev_response = prev_response,
          current_query = current_query,
          format_instructions = format_instructions
      )
      response = self.llm.invoke(prompt)

      return json_parser(response)


    except Exception as e:
      print('오류오류', e)




실행해보기

In [ ]:
aqr = AdvancedQueryRewriter()
answer = aqr.rewrite_query(
    prev_query = "뱅드림 애니메이션이 뭐야?",
    prev_response = "수 많은 카와이한 여고생들이 나와 우리의 눈과 귀를 즐겁게 해주는 초 카와이 아니매입니다.",
    current_query = "가장 유명한 캐릭터는?"
)

print(f'answer type : {type(answer)}')
print(f'answer : {answer}')

In [ ]:
tests = [
    # 1. 애니메이션 (뱅드림)
    {"prev_query": "뱅드림 애니메이션이 뭐야?", "prev_response": "여고생들이 밴드를 결성해 성장해 나가는 초 카와이한 음악 애니메이션입니다.", "current_query": "거기 나오는 보컬 중 가장 인기 있는 애는 누구야?"},
    {"prev_query": "최애의 아이 줄거리 알려줘.", "prev_response": "아이돌과 환생이라는 독특한 소재를 다룬 복수극 애니메이션입니다.", "current_query": "원작 만화 작가가 누구야?"},

    # 2. 항공
    {"prev_query": "에어버스 A380에 대해 알려줘.", "prev_response": "하늘 위의 호텔이라 불리는 현존 최대 규모의 2층짜리 여객기입니다.", "current_query": "그거 최대 몇 명까지 탈 수 있어?"},
    {"prev_query": "블랙박스가 비행기 어디에 있어?", "prev_response": "보통 사고 시 충격이 가장 적은 비행기 꼬리 부분에 설치됩니다.", "current_query": "색깔은 왜 주황색이야?"},

    # 3. 불독
    {"prev_query": "프렌치 불독 성격 어때?", "prev_response": "조용하고 다정하며 주인에 대한 애정이 깊은 성격입니다.", "current_query": "키울 때 주의해야 할 유전병도 알려줘."},
    {"prev_query": "불독은 왜 얼굴이 찌그러져 있어?", "prev_response": "과거 투견 시절 코가 낮아야 물고 있는 상태에서도 숨을 쉴 수 있었기 때문입니다.", "current_query": "잠잘 때 코를 많이 골까?"},

    # 4. 털뭉치
    {"prev_query": "고양이 헤어볼이 뭐야?", "prev_response": "고양이가 그루밍을 하면서 삼킨 털이 위속에서 뭉쳐져 다시 역류해 나오는 것입니다.", "current_query": "이거 안 생기게 하는 방법은?"},
    {"prev_query": "강아지 배냇털 미용 언제 해?", "prev_response": "보통 예방접종이 끝나는 4~5개월 이후에 하는 것이 좋습니다.", "current_query": "싹 밀어버려도 괜찮아?"},

    # 5. 고양이
    {"prev_query": "고양이가 갑자기 우다다를 왜 해?", "prev_response": "사냥 본능에 의한 에너지를 발산하거나 화장실 사용 후 기분이 좋아서 하는 행동입니다.", "current_query": "밤에 안 하게 하려면 어떻게 해?"},
    {"prev_query": "랙돌 고양이 특징이 뭐야?", "prev_response": "안아 올리면 인형처럼 몸을 맡길 정도로 온순하고 털이 부드러운 품종입니다.", "current_query": "걔네도 털 많이 빠져?"},

    # 6. 살인사건 (추리/범죄)
    {"prev_query": "화성 연쇄 살인 사건 범인 잡혔어?", "prev_response": "네, 이춘재가 진범으로 밝혀지며 오랜 미제 사건이 해결되었습니다.", "current_query": "어떤 증거 때문에 잡힌 거야?"},
    {"prev_query": "잭 더 리퍼가 누구야?", "prev_response": "1888년 영국 런던에서 발생한 연쇄 살인 사건의 미검거 범인입니다.", "current_query": "그 당시에 의심받았던 용의자들은 누구야?"},

    # 7. 피카츄
    {"prev_query": "피카츄는 왜 지우 어깨에만 있어?", "prev_response": "몬스터볼에 들어가는 것을 싫어해서 지우와 항상 함께 다니는 쪽을 선택했습니다.", "current_query": "진화하면 어떻게 돼?"},
    {"prev_query": "피카츄의 주특기 공격은 뭐야?", "prev_response": "가장 대표적인 기술은 백만볼트와 전광석화입니다.", "current_query": "그거 말고 꼬리로 때리는 기술도 있지 않아?"},

    # 8. 닌텐도
    {"prev_query": "닌텐도 스위치 살만한 가치가 있어?", "prev_response": "독점작인 젤다, 마리오 등 명작이 많아 휴대용 게임기로는 최고의 선택입니다.", "current_query": "후속 기종은 언제 나온대?"},
    {"prev_query": "동물의 숲에서 돈 빨리 버는 법.", "prev_response": "무트코인이라고 불리는 무 주식 거래나 희귀 곤충 채집이 효율적입니다.", "current_query": "무트코인 하려면 일요일에 누구 찾아가야 해?"},

    # 9. 비트코인
    {"prev_query": "비트코인 반감기가 뭐야?", "prev_response": "약 4년마다 채굴 보상이 절반으로 줄어들어 공급량이 조절되는 메커니즘입니다.", "current_query": "다음번 예정일은 정확히 언제야?"},
    {"prev_query": "사토시 나카모토가 누구야?", "prev_response": "비트코인을 만든 정체불명의 개발자 혹은 단체의 가명입니다.", "current_query": "그 사람이 가지고 있는 비트코인 수량은 얼마나 돼?"},

    # 10. 구름
    {"prev_query": "뭉게구름의 정식 명칭이 뭐야?", "prev_response": "수직으로 발달한 적운(Cumulus)이라고 부릅니다.", "current_query": "얘가 커지면 소나기가 내리는 거야?"},
    {"prev_query": "구름은 왜 하얀색이야?", "prev_response": "빛의 산란 현상 때문에 우리 눈에 하얗게 보입니다.", "current_query": "그럼 비 오기 직전에 검게 변하는 이유는?"}
]

In [ ]:
aqr = AdvancedQueryRewriter()

for i, query in enumerate(tests):
    answer = aqr.rewrite_query(
      prev_query = query['prev_query'],
      prev_response = query['prev_response'],
      current_query = query['current_query']
    )

    print(f'{i+1} 번째 시험')
    print(f'answer : {answer}')
    print()


1 번째 시험
answer : ['뱅드림 애니메이션에 등장하는 보컬 캐릭터 중에서 가장 인기 있는 캐릭터는 누구인가?']

2 번째 시험
answer : ['최애의 아이 원작 만화 작가', '최애의 아이 만화 작가 이름', '최애의 아이 원작자']

3 번째 시험
answer : ['에어버스 A380 최대 탑승 인원', '에어버스 A380 최대 몇 명까지 탈 수 있는지']

4 번째 시험
answer : ['비행기 블랙박스가 주황색인 이유', '비행기 블랙박스 색상 선정 이유']

5 번째 시험
answer : ['프렌치 불독을 키울 때 주의해야 할 유전병', '프렌치 불독에게 흔한 유전 질환과 예방 방법']

6 번째 시험
answer : ['불독은 잠잘 때 코를 많이 고는가?', '불독의 코골이 원인과 특징']

7 번째 시험
answer : ['고양이 헤어볼 예방 방법', '고양이 헤어볼이 생기지 않게 하는 방법', '고양이 털 뭉침 방지 방법']

8 번째 시험
answer : ['강아지 배냇털 미용 시 싹 밀어도 괜찮은지', '강아지 배냇털을 완전히 밀었을 때 문제점이나 주의사항']

9 번째 시험
answer : ['고양이가 밤에 우다다하지 않게 하는 방법', '고양이의 야간 에너지 발산을 줄이는 방법', '고양이가 밤에 활발하게 움직이지 않도록 하는 방법']

10 번째 시험
answer : ['랙돌 고양이 털 빠짐', '랙돌 고양이 털 빠지는 정도', '랙돌 고양이 털 관리 방법']

11 번째 시험
answer : ['이춘재가 화성 연쇄 살인 사건의 진범으로 밝혀진 결정적 증거', '이춘재가 화성 연쇄 살인 사건 범인으로 특정된 이유', '화성 연쇄 살인 사건에서 이춘재를 범인으로 지목하게 된 증거 자료']

12 번째 시험
answer : ['잭 더 리퍼 사건 당시 의심받았던 주요 용의자 목록', '1888년 런던 잭 더 리퍼 사건의 용의자들은 누구였는가']

13 번째 시험
answer : ['피카츄가 진화하면 어떻게 되는지', '피카츄가 

##이전 대화를 고려한 쿼리 재생성 + 검색기

In [34]:
class MultiQueryRetriever:
  def __init__(self, vectorstore, query_rewriter):
    self.query_rewriter = query_rewriter
    self.vectorstore = vectorstore
    self.prev_query = ""
    self.prev_response = ""

  def update_conversation(self, query:str, response:str):
    self.prev_query = query
    self.prev_response = response

  def retrieve(self, current_query:str, num_docs:int = 3):

    #1. 멀티턴, 멀티쿼리 생성
    rewritten_queries = self.query_rewriter.rewrite_query(
        self.prev_query,
        self.prev_response,
        current_query
    )
    print(f'원래 쿼리 : {current_query}')
    print(f'재생성된 쿼리 : {rewritten_queries}')

    seen_contents = set()
    all_docs = []

    print(f'rewritten queries : {type(rewritten_queries)}, {rewritten_queries}')

    for idx, rewritten_query in enumerate(rewritten_queries):
      docs = self.vectorstore.similarity_search(rewritten_query, k=num_docs)

      for doc in docs:
        if doc.page_content not in seen_contents:
          seen_contents.add(doc.page_content)
          all_docs.append(doc)

    return all_docs

In [21]:
aqr = AdvancedQueryRewriter()
mqr = MultiQueryRetriever(vectordb, aqr)

In [22]:
query = '북한 사람들의 생활상'
mqr.retrieve(query)
mqr.update_conversation(query, '북한 사람들은 서울에서 삽니다')

원래 쿼리 : 북한 사람들의 생활상
재생성된 쿼리 : ['북한 사람들의 생활상']
rewritten queries : <class 'list'>, ['북한 사람들의 생활상']


In [23]:
query = '거기서는 어떻게 사니?'
mqr.retrieve(query)

원래 쿼리 : 거기서는 어떻게 사니?
재생성된 쿼리 : ['북한 사람들이 서울에서 어떻게 생활하는지', '북한 사람들이 서울에서의 일상생활']
rewritten queries : <class 'list'>, ['북한 사람들이 서울에서 어떻게 생활하는지', '북한 사람들이 서울에서의 일상생활']


[Document(id='83c0c4f0-78cd-468d-bc3e-48b2f2385054', metadata={'producer': 'Adobe PDF Library 10.0.1', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2023-07-31T13:50:27+09:00', 'moddate': '2023-07-31T13:57:54+09:00', 'trapped': '/False', 'source': '2023_북한인권보고서.pdf', 'total_pages': 448, 'page': 194, 'page_label': '195'}, page_content='서는 대부분의 사람이 한국 영화나 드라마를 시청했다고 한다. 그리\n고 가족들이 다 함께 모여 노트텔로 한국영화를 보았다는 진술이 수\n집되었는데 한국 영화는 북한 영화와는 달리 실질적인 모습을 보여\n주어 재미있었다고 한다. 2019년에도 주변인들을 통해 한국 드라\n201\t \tUN\tDoc.\tA/HRC/WG.6/33/PRK/1\t(2019),\tpara.\t30.'),
 Document(id='6f588718-80dc-4b4e-8636-4ab12dab7f5b', metadata={'producer': 'Adobe PDF Library 10.0.1', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2023-07-31T13:50:27+09:00', 'moddate': '2023-07-31T13:57:54+09:00', 'trapped': '/False', 'source': '2023_북한인권보고서.pdf', 'total_pages': 448, 'page': 227, 'page_label': '228'}, page_content='경북도 청진시 인근까지 라고 한다. 평양시민은 이동 중 검문초소의 \n수시 검문에서도 신분증과 얼굴을 대조하기만 할 뿐, 짐이나 소지품 \n검사를 하지 

##두 클래스를 사용한 RAG 시스템 완성

In [60]:
class AdvancedConversationalRAG:
  def __init__(self, vectorstore, model : str = 'gpt-4.1', temperature : float = 0.2):

    self.query_rewriter = AdvancedQueryRewriter()
    self.retriever = MultiQueryRetriever(vectorstore = vectorstore, query_rewriter=self.query_rewriter)

    self.llm = ChatOpenAI(model_name = model, temperature = temperature)
    self.prompt_template = ChatPromptTemplate.from_template("""
     당신은 사용자 질문에 대한 정보를 제공하는 도우미입니다.

        사용자 질문: {query}

        다음 정보를 참고하여 사용자 질문에 답변하세요:
        {context}

        참고 사항:
        1. 사용자 질문에 여러 개의 질의가 포함되어 있다면, 각각에 대해 명확하게 답변하세요.
        2. 제공된 정보만을 사용하여 답변하세요. 정보가 부족하면 솔직히 모른다고 답변하세요.
        3. 답변은 한국어로 제공하세요.
        4. 제공된 정보의 원본 출처가 있다면 인용해주세요.

        답변:
    """)


  def query(self, current_query):
    #1. 문서 검색 후 context 생성
    docs = self.retriever.retrieve(current_query)
    context = "\n\n".join([doc.page_content for doc in docs])

    #2. prompt 작성
    prompt = self.prompt_template.format_messages(query = current_query, context = context)
    # print(prompt[0].content)

    #3. LLM 호출
    response = self.llm.invoke(prompt)

    #4. 대화 업데이트
    self.retriever.update_conversation(current_query, response.content)

    return {
        'query' : current_query,
        'response' : response.content,
        'retrieved_docs' : docs
    }


질문 + 후속 질문

In [61]:
rag = AdvancedConversationalRAG(vectorstore = vectordb, model = 'gpt-4.1', temperature = 1)
answer = rag.query('북한 사람들의 문화')

print(f'query : {answer['query']}')
print(f'response : {answer['response']}')
print(f'retrieved_docs : {answer['retrieved_docs']}')

원래 쿼리 : 북한 사람들의 문화
재생성된 쿼리 : ['북한 사람들의 문화']
rewritten queries : <class 'list'>, ['북한 사람들의 문화']
query : 북한 사람들의 문화
response : 북한 사람들의 문화에 대해 제공된 정보를 바탕으로 답변드리겠습니다.

1. **한국 영화와 드라마의 시청**  
   최근 몇 년 동안 양강도 혜산시를 비롯한 북한 지역에서는 많은 사람들이 한국 영화나 드라마를 시청했다고 합니다. 가족들이 함께 모여 노트텔(휴대용 매체 재생기기)로 한국 영화를 보는 경우도 있었다고 하며, 북한 영화와 달리 한국 영화는 현실적인 모습을 그려 더 재미있다는 평가가 있었습니다.

2. **외부정보 유입 경로**  
   한국 영화나 드라마 등 외부정보는 주로 해외를 오가는 유학생, 해외노동자, 밀무역 장사꾼, 그리고 고위층 또는 세관원에게 뇌물을 주는 방법을 통해 북한으로 들어온다고 합니다. 이렇게 들어온 외부정보는 지인들 사이에서 전파되어 더 많은 주민들이 접하게 되는 것으로 나타났습니다.

3. **감시와 처벌**  
   북한 당국은 인민반이나 연합지휘부를 통해 주민들의 사상 동향을 감시하고, 비사회주의 행위를 단속 및 처벌한 사례가 있습니다. 외부 문화를 접하는 것이 엄격히 통제되고 있으며, 발견될 경우 처벌이 이루어질 수 있다고 합니다.

4. **종교와 문화생활**  
   종교의 자유는 법적으로만 명시되어 있을 뿐 실제로는 보장되지 않는 것으로 보고되었으며, 북한 주민들 대부분은 종교를 접한 경험이 없는 것으로 나타났습니다. 학교 및 조직생활에서는 반종교 교육이 실시되고, 성경 소지나 선교활동도 금지되어 있습니다.

출처: UN Doc. A/HRC/WG.6/33/PRK/1 (2019), para. 30.
retrieved_docs : [Document(id='83c0c4f0-78cd-468d-bc3e-48b2f2385054', metadata={'producer': 'Adobe PDF Library 

In [62]:
answer = rag.query('그들의 종교적 활동')

print(f'query : {answer['query']}')
print(f'response : {answer['response']}')
print(f'retrieved_docs : {answer['retrieved_docs']}')

원래 쿼리 : 그들의 종교적 활동
재생성된 쿼리 : ['북한 주민들의 종교적 활동 실태', '북한에서 종교 활동이 어떻게 이루어지는지', '북한 주민들의 종교 생활과 관련된 제한 및 처벌 사례']
rewritten queries : <class 'list'>, ['북한 주민들의 종교적 활동 실태', '북한에서 종교 활동이 어떻게 이루어지는지', '북한 주민들의 종교 생활과 관련된 제한 및 처벌 사례']
query : 그들의 종교적 활동
response : 북한 주민들의 종교적 활동에 대해서는 다음과 같은 특징이 있습니다.

1. 종교 경험의 결핍  
북한 주민 대부분은 종교를 접해본 경험이 거의 없는 것으로 나타났습니다. 이는 학교나 조직생활에서 지속적으로 반종교 교육이 이루어지고 있고, 이러한 활동이 전 사회적으로 강하게 통제되고 있기 때문입니다.

2. 종교 탄압과 처벌  
성경을 소지한다거나, 선교 활동을 했다는 이유로 주민이 정치범수용소에 수감되거나 공개처형되는 사례가 있다는 증언이 다수 수집되고 있습니다. 2018년 이후에는 미신행위도 비사회주의적 행위로 간주되어 단속 및 처벌이 더욱 강화되었고, 노동교화형 또는 사형에 처해지는 경우도 있었습니다.

3. 명목상 종교의 자유  
북한의 사회주의헌법(2019) 제68조는 신앙의 자유를 보장한다고 명시하지만, “종교를 외세를 끌어들이거나 국가사회질서를 해치는데 이용할 수 없다”고 제한을 두고 있습니다. 실제로는 종교의 자유가 법률상으로만 존재하며, 주민 대다수는 종교 활동 경험이 없다고 진술하였습니다.

4. 선전용 종교시설  
북한에 몇몇 교회나 성당 등 종교시설이 있기는 하지만, 이들 대부분은 외국인을 대상으로 한 선전용 시설이라는 증언이 다수입니다. 북한 주민은 이런 시설에 출입하는 것이 금지되어 있는 것으로 나타났습니다.

5. 단속 및 처벌 사례  
예를 들어, 2017년 함경북도에서는 선교 활동을 한 혐의로 마을 주민 12명이 보위부에 구속되어 조사를 받았고, 일부는 정치범수용소에 수감, 나머지

In [63]:
answer = rag.query('이 나라의 국가사회질서에 대하여')

print(f'query : {answer['query']}')
print(f'response : {answer['response']}')
print(f'retrieved_docs : {answer['retrieved_docs']}')

원래 쿼리 : 이 나라의 국가사회질서에 대하여
재생성된 쿼리 : ['북한의 국가사회질서란 무엇인가', '북한에서 국가사회질서 유지 방식', '북한의 국가사회질서와 종교적 통제의 관계', '북한 헌법 및 법률에서 규정하는 국가사회질서', '북한 주민의 일상생활과 국가사회질서의 영향']
rewritten queries : <class 'list'>, ['북한의 국가사회질서란 무엇인가', '북한에서 국가사회질서 유지 방식', '북한의 국가사회질서와 종교적 통제의 관계', '북한 헌법 및 법률에서 규정하는 국가사회질서', '북한 주민의 일상생활과 국가사회질서의 영향']
query : 이 나라의 국가사회질서에 대하여
response : 북한의 국가사회질서에 대해서 제공된 정보를 바탕으로 정리하여 답변드립니다.

1. 국가사회질서의 통제와 사상교육  
북한 사회는 철저한 통제와 감시 체계 위에 국가사회질서가 운영되고 있습니다. 북한은 헌법상 ‘김일성-김정일주의’를 유일한 지도적 지침으로 규정하고 ‘당의 유일적 영도체계 확립의 10대 원칙’ 등 다양한 규범을 통해 주민들의 사상과 행동을 통제합니다. 일반교육보다 정치사상교육이 강조되며, 실탄사격 등 군사훈련이 교육과정에 포함되어 학생들이 의무적으로 참여해야 합니다.

2. 주민에 대한 감시와 생활검열  
국가는 인민반 제도, 생활총화 등 조직을 활용해 주민 상호 간의 감시와 통제를 강화합니다. 불순한 영상물 단속, 말반동(정권에 반하는 발언) 단속 등이 일상적으로 이루어지며, 표현·집회·결사의 자유는 매우 제한적으로 보장되고 있습니다. 또한 청년들의 휴대전화 검열, 통신 감청, 서신 검열, 복장·머리모양 검열 등이 정기적 혹은 수시로 강화되고 있습니다. 이러한 검열과 단속은 ‘제 나라의 식대로 살아야 한다’는 방침 아래 더욱 엄격하게 시행되고 있습니다.

3. 종교 및 양심의 자유 제한  
비록 헌법에는 종교의 자유가 명문화되어 있으나, 실제로는 반종교 교육이 이루어지고 외세 유입이나 사회질서 해명의 우려가 있는 종교활동은